# 数据处理

## 导入必要库

In [1]:
import pandas as pd

## 导入原始数据


In [2]:
data = pd.read_csv("源数据/UserBehavior.csv")
data.head() # 查看前5行数据


,1,2268318,2520377,pv,1511544070
0,1,2333346,2520771,pv,1511561733
1,1,2576651,149192,pv,1511572885
2,1,3830808,4181361,pv,1511593493
3,1,4365585,2520377,pv,1511596146
4,1,4606018,2735466,pv,1511616481


In [3]:
# 第一行不是标题，所以指定 header=None 并手动给列名
# 原始 timestamp 有两个坑，这里一并处理：
#   1) 它是北京时间对应的秒级时间戳，直接 unit='s' 会按 UTC 解释，整体早 8 小时
#   2) 混有约 5.5 万条时间戳越界的脏数据（最早 1902 年、最晚 2037 年），需要按官方区间过滤
start = int(pd.Timestamp('2017-11-25', tz='Asia/Shanghai').timestamp())
end = int(pd.Timestamp('2017-12-04', tz='Asia/Shanghai').timestamp()) - 1

chunks = []
n_drop = 0
for chunk in pd.read_csv(
    '源数据/UserBehavior.csv',
    header=None,
    names=['user_id', 'item_id', 'category_id', 'behavior_type', 'timestamp'],
    chunksize=1000000,
):
    # 剔除越界记录
    mask = chunk['timestamp'].between(start, end)
    n_drop += int((~mask).sum())
    chunk = chunk[mask].copy()
    # 时间戳 +8 小时，转成真正的北京时间
    chunk['datetime'] = pd.to_datetime(chunk['timestamp'] + 8 * 3600, unit='s')
    chunks.append(chunk)

print(f'剔除越界记录 {n_drop:,} 条')

data = pd.concat(chunks, ignore_index=True)
del chunks
print('时间范围:', data['datetime'].min(), '~', data['datetime'].max())
print('总行数:', len(data))
data.head() # 查看前5行数据


剔除越界记录 55,576 条


时间范围: 2017-11-25 00:00:00 ~ 2017-12-03 23:59:59
总行数: 100095231


,user_id,item_id,category_id,behavior_type,timestamp,datetime
0,1,2268318,2520377,pv,1511544070,2017-11-25 01:21:10
1,1,2333346,2520771,pv,1511561733,2017-11-25 06:15:33
2,1,2576651,149192,pv,1511572885,2017-11-25 09:21:25
3,1,3830808,4181361,pv,1511593493,2017-11-25 15:04:53
4,1,4365585,2520377,pv,1511596146,2017-11-25 15:49:06


### 数据信息

In [4]:
data.info() # 查看数据信息

<class 'pandas.DataFrame'>
RangeIndex: 100095231 entries, 0 to 100095230
Data columns (total 6 columns):
 #   Column         Dtype         
---  ------         -----         
 0   user_id        int64         
 1   item_id        int64         
 2   category_id    int64         
 3   behavior_type  str           
 4   timestamp      int64         
 5   datetime       datetime64[us]
dtypes: datetime64[us](1), int64(4), str(1)
memory usage: 4.5 GB


In [5]:
data.isnull().sum() # 查看缺失值数量

user_id          0
item_id          0
category_id      0
behavior_type    0
timestamp        0
datetime         0
dtype: int64

## 看得出没有缺失数据，也不需要标准化单位化数据，所以不需要处理缺失值

已完成的清洗：

- 补列名（原始文件没有表头）
- 剔除时间戳越界的脏数据（约 5.5 万条）
- `datetime` 按时区修正为北京时间（原写法会比真实时间早 8 小时）

In [6]:
data.to_csv(f'data/UserBehavior_clean.csv', index=False, encoding='utf-8-sig')

In [7]:
newdata = pd.read_csv(f'data/UserBehavior_clean.csv')
newdata.head() # 新数据比原始数据多了一列datetime

,user_id,item_id,category_id,behavior_type,timestamp,datetime
0,1,2268318,2520377,pv,1511544070,2017-11-25 01:21:10
1,1,2333346,2520771,pv,1511561733,2017-11-25 06:15:33
2,1,2576651,149192,pv,1511572885,2017-11-25 09:21:25
3,1,3830808,4181361,pv,1511593493,2017-11-25 15:04:53
4,1,4365585,2520377,pv,1511596146,2017-11-25 15:49:06
